# Frames per animal and working range in the field (T7a, T7b)

The seven reef deployments behind P1 §4.1. **Framing constraint** (`fishsense_cscw/field.py`):
P1 derives "≈10 frames per animal" and "beyond 1 m" from its own analysis; nothing on disk
shows divers were told either. These are shortfalls against what the estimator needs,
not non-compliance. The pose instruction (≤15°) is the one divers were given, and it
needs the mask-based pose estimator (T7c).

In [1]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.precision", 2)

from fishsense_cscw import field
df = field.load_field()
per = field.frames_per_animal(df)
print(f"{len(df)} measurements, {per.size} animals, {df.dive_id.nunique()} dives, {df.camera_id.nunique()} cameras")
print(f"frames per animal: median {per.median():.0f}, max {per.max()}, >=10: {(per >= 10).sum()}, "
      f"exactly one: {(per == 1).sum()} ({(per == 1).mean():.0%})")
per.value_counts().sort_index().rename("animals").to_frame().T

162 measurements, 73 animals, 7 dives, 6 cameras
frames per animal: median 2, max 8, >=10: 0, exactly one: 31 (42%)


,1,2,3,4,5,6,7,8
animals,31,17,16,4,1,1,2,1


In [2]:
pd.Series(field.range_bands(df)).round(2).to_frame("laser range, measured frames").T

,n,p10,p50,p90,below_1m,below_1m_pct,in_table1_range,in_table1_range_pct,below_2m_pct
"laser range, measured frames",162.0,0.76,1.49,2.71,35.0,21.6,43.0,26.54,73.46


In [3]:
df.groupby("dive_id").agg(animals=("animal", "nunique"), frames=("animal", "size"),
                          range_p50=("range_m", "median"),
                          below_1m=("range_m", lambda r: (r < 1).mean()))

,animals,frames,range_p50,below_1m
dive_id,,,,
279,4,8,1.18,0.25
341,11,27,0.86,0.74
347,11,36,1.58,0.28
349,13,37,1.48,0.03
383,5,7,1.49,0.14
465,18,31,1.82,0.00
471,11,16,1.87,0.06


## Result

- **No animal reaches the ~10 frames a stable p90 needs; 42 % have exactly one.** Every
  field "p90" is a sample maximum, and for 31 of 73 animals it is a single photograph.
  P1's handoff had median 2 / max 8; the singleton share is new.
- **22 % of measured frames are inside 1 m**, the regime P1 advises against — and one
  deployment (dive 341) is 74 % inside 1 m, so it is a per-diver or per-site habit rather
  than a fleet property.
- **P1's Table 1 advertises a 2–5 m working range; only 27 % of field measurements fall
  in it** (median 1.49 m, max 3.9 m, 73 % below 2 m). That is a discrepancy inside P1,
  worth passing to its author.

Limit: these are *measured* frames. Frames where the laser missed or that were never
labelled are absent, so frames-per-animal is a count of usable frames — which is the
quantity p90 depends on, but not a count of what divers shot.